# 1. 데이터 로딩 & 기본 검증
## 1.0 환경 설정

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_DIR = Path("./data")
TRAIN_PATH = DATA_DIR / "train" / "train.csv"
TEST_DIR = DATA_DIR / "test"
TEST_PATHS = sorted(TEST_DIR.glob("TEST_*.csv"))

print("TRAIN_PATH:", TRAIN_PATH)
print("N_TEST_FILES:", len(TEST_PATHS))
print("TEST_FILES:", [p.name for p in TEST_PATHS])

TRAIN_PATH: data\train\train.csv
N_TEST_FILES: 10
TEST_FILES: ['TEST_00.csv', 'TEST_01.csv', 'TEST_02.csv', 'TEST_03.csv', 'TEST_04.csv', 'TEST_05.csv', 'TEST_06.csv', 'TEST_07.csv', 'TEST_08.csv', 'TEST_09.csv']


## 1.1 데이터 로드

In [2]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raws = {p.stem: pd.read_csv(p) for p in TEST_PATHS}

print("train_raw:", train_raw.shape)
for k, v in test_raws.items():
    print(k, v.shape)

train_raw.head()

train_raw: (102676, 3)
TEST_00 (5404, 3)
TEST_01 (5404, 3)
TEST_02 (5404, 3)
TEST_03 (5404, 3)
TEST_04 (5404, 3)
TEST_05 (5404, 3)
TEST_06 (5404, 3)
TEST_07 (5404, 3)
TEST_08 (5404, 3)
TEST_09 (5404, 3)


,영업일자,영업장명_메뉴명,매출수량
0,2023-01-01,느티나무 셀프BBQ_1인 수저세트,0
1,2023-01-02,느티나무 셀프BBQ_1인 수저세트,0
2,2023-01-03,느티나무 셀프BBQ_1인 수저세트,0
3,2023-01-04,느티나무 셀프BBQ_1인 수저세트,0
4,2023-01-05,느티나무 셀프BBQ_1인 수저세트,0


## 1.2 스키마 검증

In [3]:
REQUIRED_COLS = ["영업일자", "영업장명_메뉴명", "매출수량"]

missing = [c for c in REQUIRED_COLS if c not in train_raw.columns]
assert len(missing) == 0, f"train.csv missing columns: {missing}"

for name, df in test_raws.items():
    missing_t = [c for c in REQUIRED_COLS if c not in df.columns]
    assert len(missing_t) == 0, f"{name} missing columns: {missing_t}"

print("✅ schema ok")


✅ schema ok


# 2. 기본 전처리
## 2.1 기본 전처리 함수

In [4]:
def basic_preprocess(
    df: pd.DataFrame,
    date_col: str = "영업일자",
    key_col: str = "영업장명_메뉴명",
    y_col: str = "매출수량",
    pad_calendar: bool = True,
) -> pd.DataFrame:
    out = df.copy()

    # 1) 날짜 파싱
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    if out[date_col].isna().any():
        bad = out[out[date_col].isna()].head(10)
        raise ValueError(f"[ERROR] 날짜 파싱 실패(NaT) 존재. 예시:\n{bad}")

    # 2) 영업장/메뉴 분리
    split = out[key_col].astype(str).str.split("_", n=1, expand=True)
    if split.shape[1] < 2:
        bad = out[~out[key_col].astype(str).str.contains("_")].head(10)
        raise ValueError(f"[ERROR] '{key_col}'에 '_' 없는 값 존재. 예시:\n{bad}")

    out["영업장명"] = split[0]
    out["메뉴명"] = split[1]

    # 3) 매출수량 타입
    out[y_col] = pd.to_numeric(out[y_col], errors="coerce")
    if out[y_col].isna().any():
        bad = out[out[y_col].isna()].head(10)
        raise ValueError(f"[ERROR] 매출수량 NaN 존재. 예시:\n{bad}")

    # 4) 음수 보정 (drop 대신 권장)
    out[y_col] = out[y_col].clip(lower=0)

    # 5) (필수) 중복행 집계: (키, 날짜) 유일하게
    out = (
        out.groupby([key_col, date_col, "영업장명", "메뉴명"], as_index=False)[y_col]
           .sum()
    )

    # 6) (필수) 캘린더 패딩: 업장×메뉴별 연속 날짜로 만들고 누락=0
    if pad_calendar:
        padded = []
        for (store, menu), g in out.groupby(["영업장명", "메뉴명"], sort=False):
            g = g.sort_values(date_col)
            full_dates = pd.date_range(g[date_col].min(), g[date_col].max(), freq="D")

            gg = g.set_index(date_col).reindex(full_dates)
            gg.index.name = date_col
            gg = gg.reset_index()

            gg["영업장명"] = store
            gg["메뉴명"] = menu
            gg[key_col] = store + "_" + menu
            gg[y_col] = gg[y_col].fillna(0.0)

            padded.append(gg)

        out = pd.concat(padded, ignore_index=True)

    # 7) 정렬
    out = out.sort_values(["영업장명", "메뉴명", date_col]).reset_index(drop=True)
    return out

## 2.2 전처리 적용

In [5]:
train = basic_preprocess(train_raw)
tests = {name: basic_preprocess(df) for name, df in test_raws.items()}

print("train:", train.shape)
print("tests:", {k: v.shape for k, v in tests.items()})
train.head()


train: (102676, 5)
tests: {'TEST_00': (5404, 5), 'TEST_01': (5404, 5), 'TEST_02': (5404, 5), 'TEST_03': (5404, 5), 'TEST_04': (5404, 5), 'TEST_05': (5404, 5), 'TEST_06': (5404, 5), 'TEST_07': (5404, 5), 'TEST_08': (5404, 5), 'TEST_09': (5404, 5)}


,영업일자,영업장명_메뉴명,영업장명,메뉴명,매출수량
0,2023-01-01,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
1,2023-01-02,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
2,2023-01-03,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
3,2023-01-04,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
4,2023-01-05,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0


# 3. 외부 데이터 로드 (Weather)

In [6]:
# =========================================================
# [Weather Load] temperature / rain data
# - ./data/temperature.csv
# - ./data/rain.csv 또는 ./data/rain_day.csv
# =========================================================
import pandas as pd
import numpy as np
from pathlib import Path

WEATHER_DIR = Path("./data")

TEMP_PATH = WEATHER_DIR / "temperature.csv"
RAIN_PATH = WEATHER_DIR / "rain.csv"
RAIN_DAY_PATH = WEATHER_DIR / "rain_day.csv"

temp_df = pd.read_csv(TEMP_PATH)

# rain 파일명 2개 중 존재하는 것 자동 선택
if RAIN_PATH.exists():
    rain_df = pd.read_csv(RAIN_PATH)
elif RAIN_DAY_PATH.exists():
    rain_df = pd.read_csv(RAIN_DAY_PATH)
else:
    raise FileNotFoundError("rain.csv 또는 rain_day.csv를 ./data/ 아래에 두세요.")

print("temp_df:", temp_df.shape, "columns:", temp_df.columns.tolist())
print("rain_df:", rain_df.shape, "columns:", rain_df.columns.tolist())


temp_df: (532, 4) columns: ['날짜', '평균기온', '최저기온', '최고기온']
rain_df: (532, 2) columns: ['날짜', '강수량']


# 4. Feature Engineering
## 4.1 날짜 기반 Features

In [7]:
import numpy as np
import pandas as pd
import holidays

def add_date_features(
    df: pd.DataFrame,
    date_col: str = "영업일자",
    store_col: str = "영업장명",
    add_day_of_year: bool = True,
    add_cyclical_dow: bool = True,
    store_peak_months: dict | None = None,   # ✅ 업장별 성수기 맵(SET)
) -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce").dt.normalize()

    # 1) 기본 날짜 피처
    out["year"] = out[date_col].dt.year.astype(np.int16)
    out["month"] = out[date_col].dt.month.astype(np.int8)
    out["day"] = out[date_col].dt.day.astype(np.int8)
    out["weekday"] = out[date_col].dt.weekday.astype(np.int8)
    out["is_weekend"] = (out["weekday"] >= 5).astype(np.int8)

    # 2) 주기성 및 연차 피처
    if add_day_of_year:
        out["day_of_year"] = out[date_col].dt.dayofyear.astype(np.int16)
    if add_cyclical_dow:
        dow = out["weekday"].astype(np.float32)
        out["sin_dow"] = np.sin(2 * np.pi * dow / 7).astype(np.float32)
        out["cos_dow"] = np.cos(2 * np.pi * dow / 7).astype(np.float32)

    # 3) 한국 공휴일 + 공휴일 거리 (FIX: DatetimeIndex/Series 혼동 + apply 제거)
    years = sorted(out[date_col].dt.year.unique().tolist())
    kr_holidays = holidays.country_holidays("KR", years=years)

    # 공휴일 날짜들을 numpy datetime64[D] 배열로 고정
    holiday_days = np.array(
        sorted(pd.to_datetime(list(kr_holidays.keys())).normalize()),
        dtype="datetime64[D]"
    )
    curr_days = out[date_col].values.astype("datetime64[D]")

    # 다음/이전 공휴일 인덱스 (벡터화)
    idx_next = np.searchsorted(holiday_days, curr_days, side="left")
    idx_prev = np.searchsorted(holiday_days, curr_days, side="right") - 1

    # 범위 밖 처리(없으면 ±30일로 캡)
    next_days = np.where(
        idx_next < len(holiday_days),
        holiday_days[idx_next],
        curr_days + np.timedelta64(30, "D"),
    )
    prev_days = np.where(
        idx_prev >= 0,
        holiday_days[idx_prev],
        curr_days - np.timedelta64(30, "D"),
    )

    out["is_holiday"] = np.isin(curr_days, holiday_days).astype(np.int8)
    out["days_to_next_holiday"] = (
        (next_days - curr_days).astype("timedelta64[D]").astype(np.int16)
    )
    out["days_from_prev_holiday"] = (
        (curr_days - prev_days).astype("timedelta64[D]").astype(np.int16)
    )

    # 연휴 인접 (±1/±2)
    out["is_holiday_near_1"] = (
        (out["days_to_next_holiday"] <= 1) | (out["days_from_prev_holiday"] <= 1)
    ).astype(np.int8)
    out["is_holiday_near_2"] = (
        (out["days_to_next_holiday"] <= 2) | (out["days_from_prev_holiday"] <= 2)
    ).astype(np.int8)

    # 4) ✅ 업장별 성수기(peak month)
    # store_peak_months: {"포레스트릿": {1,2,12}, ...}
    if store_peak_months is not None and store_col in out.columns:
        # 성능 위해 apply 최소화: month 벡터 + map
        store_to_peak = store_peak_months
        months = out["month"].astype(np.int16).to_numpy()
        stores = out[store_col].astype(str).to_numpy()

        # store별 set lookup
        out["store_is_peak_month"] = np.fromiter(
            (1 if int(m) in store_to_peak.get(s, set()) else 0 for s, m in zip(stores, months)),
            dtype=np.int8,
            count=len(out),
        )
    else:
        out["store_is_peak_month"] = 0

    return out


# 적용
train_f = add_date_features(train, store_peak_months=None)
tests_f = {k: add_date_features(v, store_peak_months=None) for k, v in tests.items()}

# QC
def qc_after_date_features(df, name="df"):
    must = ["영업일자", "영업장명_메뉴명", "영업장명", "메뉴명", "매출수량",
            "weekday", "is_weekend", "month"]
    missing = [c for c in must if c not in df.columns]
    assert not missing, f"[{name}] missing after date features: {missing}"
    assert df["영업일자"].isna().sum() == 0, f"[{name}] date has NaT"
    assert np.isfinite(df["매출수량"].to_numpy()).all(), f"[{name}] y has non-finite"
    print(f"✅ date-features QC PASS: {name}")

qc_after_date_features(train_f, "train")
for k, v in tests_f.items():
    qc_after_date_features(v, k)


✅ date-features QC PASS: train
✅ date-features QC PASS: TEST_00
✅ date-features QC PASS: TEST_01
✅ date-features QC PASS: TEST_02
✅ date-features QC PASS: TEST_03
✅ date-features QC PASS: TEST_04
✅ date-features QC PASS: TEST_05
✅ date-features QC PASS: TEST_06
✅ date-features QC PASS: TEST_07
✅ date-features QC PASS: TEST_08
✅ date-features QC PASS: TEST_09


## 4.2 메뉴 기반 Features

In [8]:
import re
import numpy as np
import pandas as pd

def _normalize_menu_name(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str)
    s = s.str.strip().str.lower()
    s = s.str.replace(r"[^0-9a-z가-힣]+", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

def classify_menu_type_series(menu_series: pd.Series) -> pd.Series:
    name = _normalize_menu_name(menu_series)

    def has_any(keywords):
        pat = r"(?:^| )(" + "|".join(map(re.escape, keywords)) + r")(?: |$)"
        return name.str.contains(pat, regex=True, na=False)

    set_kw = ["세트","코스","정식","패키지","모둠","플래터","콤보","단체","set","course","combo","platter"]
    alcohol_kw = ["소주","맥주","와인","막걸리","칵테일","위스키","사케","하이볼","beer","wine","whisky","sake","highball"]
    drink_kw = ["커피","아메리카노","라떼","에스프레소","카푸치노","차","티","주스","에이드","스무디",
                "콜라","사이다","탄산","생수","음료","water",
                "coffee","americano","latte","espresso","tea","juice","ade","smoothie","soda"]
    food_kw = ["밥","국","찌개","탕","전골","면","라면","우동","파스타","피자","버거","샐러드","덮밥","비빔","볶음","구이","튀김","전",
               "스테이크","돈까스","카레","김밥","떡볶이","갈비","삼겹","목살","불고기","치킨"]

    is_set = has_any(set_kw)
    is_alcohol = has_any(alcohol_kw)
    is_drink = has_any(drink_kw)
    is_food = has_any(food_kw)

    # 우선순위: 세트 > 주류 > 음료 > 음식 > 기타
    out = np.select([is_set, is_alcohol, is_drink, is_food], [4, 3, 2, 1], default=0).astype(np.int8)
    return pd.Series(out, index=menu_series.index, name="menu_type")

def add_menu_features(
    df: pd.DataFrame,
    menu_col: str = "메뉴명",
    key_col: str = "영업장명_메뉴명",
) -> pd.DataFrame:
    out = df.copy()

    # 필수 컬럼 체크 (전처리 결과 기준)
    if menu_col not in out.columns:
        # 안전장치: 혹시 메뉴명 컬럼이 없으면 key에서 복원
        split = out[key_col].astype(str).str.split("_", n=1, expand=True)
        if split.shape[1] < 2:
            bad = out[~out[key_col].astype(str).str.contains("_")].head(5)
            raise ValueError(f"[menu_min] cannot derive menu from '{key_col}'. Examples:\n{bad}")
        out[menu_col] = split[1]

    out["menu_type"] = classify_menu_type_series(out[menu_col]).astype(np.int8)

    # menu_form은 정말 필요할 때만(세트 여부만)
    m_norm = _normalize_menu_name(out[menu_col])
    set_hint = m_norm.str.contains(r"(?:^| )(세트|코스|정식|패키지|모둠|플래터|콤보|단체|set|course|combo|platter)(?: |$)",
                                   regex=True, na=False)
    out["menu_form"] = np.where((out["menu_type"] == 4) | set_hint, 1, 0).astype(np.int8)

    return out

# 적용
train_f = add_menu_features(train_f)
tests_f = {k: add_menu_features(v) for k, v in tests_f.items()}

# 메뉴 기반 QC
def qc_after_menu_features(df, name="df"):
    must = ["메뉴명", "menu_type", "menu_form"]
    missing = [c for c in must if c not in df.columns]
    assert not missing, f"[{name}] missing menu features: {missing}"
    # 값 범위 sanity
    assert df["menu_type"].between(0, 4).all(), f"[{name}] menu_type out of range"
    assert df["menu_form"].isin([0,1]).all(), f"[{name}] menu_form not binary"
    print(f"✅ menu-features QC PASS: {name}")

qc_after_menu_features(train_f, "train")
for k, v in tests_f.items():
    qc_after_menu_features(v, k)

C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return name.str.contains(pat, regex=True, na=False)
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:56: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  set_hint = m_norm.str.contains(r"(?:^| )(세트|코스|정식|패키지|모둠|플래터|콤보|단체|set|course|combo|platter)(?: |$)",
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return name.str.contains(pat, regex=True, na=False)
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:56: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.e

✅ menu-features QC PASS: train
✅ menu-features QC PASS: TEST_00
✅ menu-features QC PASS: TEST_01
✅ menu-features QC PASS: TEST_02
✅ menu-features QC PASS: TEST_03
✅ menu-features QC PASS: TEST_04
✅ menu-features QC PASS: TEST_05
✅ menu-features QC PASS: TEST_06
✅ menu-features QC PASS: TEST_07
✅ menu-features QC PASS: TEST_08
✅ menu-features QC PASS: TEST_09


C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:56: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  set_hint = m_norm.str.contains(r"(?:^| )(세트|코스|정식|패키지|모둠|플래터|콤보|단체|set|course|combo|platter)(?: |$)",
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return name.str.contains(pat, regex=True, na=False)
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:56: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  set_hint = m_norm.str.contains(r"(?:^| )(세트|코스|정식|패키지|모둠|플래터|콤보|단체|set|course|combo|platter)(?: |$)",
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:17: UserWarning: This pattern is interpreted as a regular expression, and has m

## 4.3 업장 기반 Features

In [9]:
import numpy as np
import pandas as pd

def add_store_features(
    target_df: pd.DataFrame,
    history_df: pd.DataFrame | None = None,
    store_col: str = "영업장명",
    key_col: str = "영업장명_메뉴명",
    date_col: str = "영업일자",
    y_col: str = "매출수량",
    lookback: int = 28,
    min_consecutive_days: int = 3,
) -> pd.DataFrame:
    def _ensure_store(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        if store_col not in out.columns:
            split = out[key_col].astype(str).str.split("_", n=1, expand=True)
            out[store_col] = split[0]
        return out

    target_df = _ensure_store(target_df)
    target_df[date_col] = pd.to_datetime(target_df[date_col], errors="coerce").dt.normalize()
    target_df[y_col] = pd.to_numeric(target_df[y_col], errors="coerce").fillna(0).clip(lower=0).astype(np.float32)

    # ✅ 핵심: daily는 history_df가 있으면 history_df로만 만든다 (중복 concat 금지)
    source = target_df if history_df is None else _ensure_store(history_df)
    source = source.copy()
    source[date_col] = pd.to_datetime(source[date_col], errors="coerce").dt.normalize()
    source[y_col] = pd.to_numeric(source[y_col], errors="coerce").fillna(0).clip(lower=0).astype(np.float32)

    daily = (
        source.groupby([store_col, date_col], as_index=False)[y_col]
              .sum()
              .rename(columns={y_col: "store_daily_sales"})
              .sort_values([store_col, date_col])
              .reset_index(drop=True)
    )

    daily["is_zero_day"] = (daily["store_daily_sales"] <= 0).astype(np.int8)
    grp = daily[store_col].astype("string")
    is_zero = daily["is_zero_day"]

    seg = (is_zero == 0).groupby(grp, sort=False).cumsum()
    daily["store_zero_streak_len"] = is_zero.groupby([grp, seg], sort=False).cumsum().astype(np.int16)

    zero_shift = (
        daily.groupby(grp, sort=False)["is_zero_day"]
             .shift(1).fillna(0).astype(np.float32)
    )
    ratio_col = f"store_zero_ratio_{lookback}"
    daily[ratio_col] = (
        zero_shift.groupby(grp, sort=False)
                  .rolling(lookback, min_periods=1).mean()
                  .reset_index(level=0, drop=True)
                  .astype(np.float32)
    )

    daily["store_is_closed"] = (daily["store_zero_streak_len"] >= min_consecutive_days).astype(np.int8)

    out = target_df.drop(columns=[c for c in ["store_zero_streak_len", ratio_col, "store_is_closed"] if c in target_df.columns])
    out = out.merge(
        daily[[store_col, date_col, "store_zero_streak_len", ratio_col, "store_is_closed"]],
        on=[store_col, date_col],
        how="left",
        validate="many_to_one",
    )
    out["store_zero_streak_len"] = out["store_zero_streak_len"].fillna(0).astype(np.int16)
    out[ratio_col] = out[ratio_col].fillna(0.0).astype(np.float32)
    out["store_is_closed"] = out["store_is_closed"].fillna(0).astype(np.int8)
    return out


In [10]:
import numpy as np
import pandas as pd

def qc_after_store_features(df: pd.DataFrame, name="df", lookback: int = 28):
    ratio_col = f"store_zero_ratio_{lookback}"
    must = ["영업장명", "영업일자", "store_zero_streak_len", ratio_col, "store_is_closed"]

    # 1) 필수 컬럼 존재
    missing = [c for c in must if c not in df.columns]
    assert not missing, f"[{name}] missing columns: {missing}"

    # 2) suffix 컬럼 체크 (_x/_y 있으면 merge 중복 흔적)
    bad_suffix = [c for c in df.columns if c.endswith("_x") or c.endswith("_y")]
    assert not any("store_" in c for c in bad_suffix), f"[{name}] suffix columns exist: {bad_suffix}"

    # 3) NaN/inf 체크
    assert df["영업일자"].isna().sum() == 0, f"[{name}] date has NaT"
    assert np.isfinite(df["store_zero_streak_len"].to_numpy()).all(), f"[{name}] streak has non-finite"
    assert np.isfinite(df[ratio_col].to_numpy()).all(), f"[{name}] ratio has non-finite"
    assert np.isfinite(df["store_is_closed"].to_numpy()).all(), f"[{name}] closed has non-finite"

    # 4) 값 범위 sanity
    assert (df["store_zero_streak_len"] >= 0).all(), f"[{name}] negative streak exists"
    assert df[ratio_col].between(0, 1).all(), f"[{name}] ratio out of [0,1]"
    assert df["store_is_closed"].isin([0, 1]).all(), f"[{name}] store_is_closed not binary"

    # 5) 간단 요약(확인용)
    print(f"✅ store-features QC PASS: {name}")
    print("   streak max:", int(df["store_zero_streak_len"].max()))
    print("   ratio mean:", float(df[ratio_col].mean()))
    print("   closed rate:", float(df["store_is_closed"].mean()))

# ✅ [추가] 2.3 업장 피처 적용 (QC 전에 반드시 실행)
train_f = add_store_features(train_f, history_df=train)  # reference는 train(패딩된 원본) 권장
tests_f = {k: add_store_features(v, history_df=train) for k, v in tests_f.items()}

qc_after_store_features(train_f, "train", lookback=28)
for k, v in tests_f.items():
    qc_after_store_features(v, k, lookback=28)


✅ store-features QC PASS: train
   streak max: 123
   ratio mean: 0.10157594084739685
   closed rate: 0.05877712415754412
✅ store-features QC PASS: TEST_00
   streak max: 0
   ratio mean: 0.0
   closed rate: 0.0
✅ store-features QC PASS: TEST_01
   streak max: 0
   ratio mean: 0.0
   closed rate: 0.0
✅ store-features QC PASS: TEST_02
   streak max: 0
   ratio mean: 0.0
   closed rate: 0.0
✅ store-features QC PASS: TEST_03
   streak max: 0
   ratio mean: 0.0
   closed rate: 0.0
✅ store-features QC PASS: TEST_04
   streak max: 0
   ratio mean: 0.0
   closed rate: 0.0
✅ store-features QC PASS: TEST_05
   streak max: 0
   ratio mean: 0.0
   closed rate: 0.0
✅ store-features QC PASS: TEST_06
   streak max: 0
   ratio mean: 0.0
   closed rate: 0.0
✅ store-features QC PASS: TEST_07
   streak max: 0
   ratio mean: 0.0
   closed rate: 0.0
✅ store-features QC PASS: TEST_08
   streak max: 0
   ratio mean: 0.0
   closed rate: 0.0
✅ store-features QC PASS: TEST_09
   streak max: 0
   ratio mean: 0.

## 4.4 날씨 기반 Features

In [ ]:
import pandas as pd
import numpy as np

def _pick_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    return None

def _standardize_weather(temp_df: pd.DataFrame, rain_df: pd.DataFrame):
    t = temp_df.copy()
    r = rain_df.copy()

    # 날짜 컬럼 찾기
    t_date = _pick_col(t, ["date", "dt", "일자", "날짜", "영업일자"])
    r_date = _pick_col(r, ["date", "dt", "일자", "날짜", "영업일자"])
    if t_date is None or r_date is None:
        raise ValueError("temperature/rain 데이터에서 날짜 컬럼을 찾지 못했습니다. (date/일자/날짜/영업일자 중 하나 필요)")

    t[t_date] = pd.to_datetime(t[t_date], errors="coerce").dt.normalize()
    r[r_date] = pd.to_datetime(r[r_date], errors="coerce").dt.normalize()
    t = t[t[t_date].notna()].copy()
    r = r[r[r_date].notna()].copy()

    # 온도 컬럼 후보: 평균/최저/최고
    tavg = _pick_col(t, ["tavg", "avg_temp", "mean_temp", "평균기온", "평균온도", "기온"])
    tmin = _pick_col(t, ["tmin", "min_temp", "최저기온", "최저온도"])
    tmax = _pick_col(t, ["tmax", "max_temp", "최고기온", "최고온도"])

    # 비(강수) 컬럼 후보
    rain = _pick_col(r, ["rain", "precip", "precipitation", "강수량", "일강수량"])

    if tavg is None and (tmin is None or tmax is None):
        raise ValueError("temperature 데이터에서 평균기온(tavg) 또는 최저/최고(tmin/tmax) 컬럼을 찾지 못했습니다.")
    if rain is None:
        raise ValueError("rain 데이터에서 강수량 컬럼(rain/precip/강수량)을 찾지 못했습니다.")

    # 숫자화
    for c in [tavg, tmin, tmax]:
        if c is not None:
            t[c] = pd.to_numeric(t[c], errors="coerce")
    r[rain] = pd.to_numeric(r[rain], errors="coerce")

    # 표준 스키마로 정리
    out_t = pd.DataFrame({"영업일자": t[t_date]})
    if tavg is not None:
        out_t["temp_avg"] = t[tavg]
    else:
        out_t["temp_avg"] = (t[tmin] + t[tmax]) / 2.0

    out_t["temp_min"] = t[tmin] if tmin is not None else np.nan
    out_t["temp_max"] = t[tmax] if tmax is not None else np.nan

    out_r = pd.DataFrame({"영업일자": r[r_date], "rain_mm": r[rain]})

    # 날짜별 1행으로 (혹시 중복 있으면 평균/합으로 정리)
    out_t = out_t.groupby("영업일자", as_index=False).mean(numeric_only=True)
    out_r = out_r.groupby("영업일자", as_index=False).sum(numeric_only=True)

    weather = out_t.merge(out_r, on="영업일자", how="outer").sort_values("영업일자").reset_index(drop=True)

    # 파생
    weather["temp_range"] = (weather["temp_max"] - weather["temp_min"]).astype(np.float32)
    weather["is_rain"] = (weather["rain_mm"].fillna(0) > 0).astype(np.int8)
    weather["rain_log1p"] = np.log1p(weather["rain_mm"].fillna(0).clip(lower=0)).astype(np.float32)

    return weather

def _build_climatology(weather: pd.DataFrame):
    w = weather.copy()
    w["mmdd"] = w["영업일자"].dt.strftime("%m-%d")
    clim = (
        w.groupby("mmdd", as_index=False)
         .agg(
            temp_avg=("temp_avg", "mean"),
            temp_min=("temp_min", "mean"),
            temp_max=("temp_max", "mean"),
            rain_mm=("rain_mm", "mean"),
            rain_log1p=("rain_log1p", "mean"),   # ✅ 추가
            temp_range=("temp_range", "mean"),
            is_rain=("is_rain", "mean"),
         )
    )
    return clim

def add_weather_features(
    df: pd.DataFrame,
    weather: pd.DataFrame,
    climatology: pd.DataFrame,
    date_col: str = "영업일자",
) -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce").dt.normalize()

    # 1) 관측 merge
    out = out.merge(weather, left_on=date_col, right_on="영업일자", how="left", suffixes=("", "_wx"))
    out.drop(columns=["영업일자_wx"], inplace=True, errors="ignore")

    # 2) climatology(mm-dd) fallback
    out["mmdd"] = out[date_col].dt.strftime("%m-%d")
    out = out.merge(climatology, on="mmdd", how="left", suffixes=("", "_clim"))

    fill_cols = ["temp_avg","temp_min","temp_max","rain_mm","rain_log1p","temp_range","is_rain"]  # ✅ rain_log1p 포함
    for c in fill_cols:
        out[c] = out[c].fillna(out[f"{c}_clim"])

    # 정리
    out.drop(columns=[f"{c}_clim" for c in fill_cols] + ["mmdd"], inplace=True, errors="ignore")

    # 타입/안전
    out["rain_mm"] = pd.to_numeric(out["rain_mm"], errors="coerce").fillna(0.0).clip(lower=0.0).astype(np.float32)
    out["rain_log1p"] = pd.to_numeric(out["rain_log1p"], errors="coerce").fillna(np.log1p(out["rain_mm"])).clip(lower=0.0).astype(np.float32)

    for c in ["temp_avg","temp_min","temp_max","temp_range"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0).astype(np.float32)

    out["is_rain"] = pd.to_numeric(out["is_rain"], errors="coerce").fillna(0.0).clip(0.0, 1.0).astype(np.float32)

    return out


# --- build once ---
WEATHER_DAILY = _standardize_weather(temp_df, rain_df)
WEATHER_CLIM = _build_climatology(WEATHER_DAILY)

print("✅ WEATHER_DAILY:", WEATHER_DAILY.shape, WEATHER_DAILY.columns.tolist())
print("✅ WEATHER_CLIM:", WEATHER_CLIM.shape, WEATHER_CLIM.columns.tolist())

# =========================================================
# [PATCH 1] APPLY WEATHER FEATURES to train_f / tests_f
# - 학습/검증에도 날씨가 들어가도록 반드시 여기서 적용
# - 위치: WEATHER_DAILY/WEATHER_CLIM 만든 직후
# =========================================================
train_f = add_weather_features(train_f, weather=WEATHER_DAILY, climatology=WEATHER_CLIM)
tests_f = {k: add_weather_features(v, weather=WEATHER_DAILY, climatology=WEATHER_CLIM) for k, v in tests_f.items()}

print("✅ weather applied to train_f/tests_f")
print("NaN check:", train_f[["temp_avg", "rain_mm", "is_rain"]].isna().sum().to_dict())


✅ WEATHER_DAILY: (532, 8) ['영업일자', 'temp_avg', 'temp_min', 'temp_max', 'rain_mm', 'temp_range', 'is_rain', 'rain_log1p']
✅ WEATHER_CLIM: (366, 8) ['mmdd', 'temp_avg', 'temp_min', 'temp_max', 'rain_mm', 'rain_log1p', 'temp_range', 'is_rain']
✅ weather applied to train_f/tests_f
NaN check: {'temp_avg': 0, 'rain_mm': 0, 'is_rain': 0}


## 4.5 시계열 Features

In [12]:
## 2.4 기타 (STEP1 버전: 학습/재귀추론 구조 통일용)  [PATCHED COMPLETE]
import numpy as np
import pandas as pd

def add_timeseries_features_step1(
    df: pd.DataFrame,
    train_reference: pd.DataFrame,
    key_col: str = "영업장명_메뉴명",
    date_col: str = "영업일자",
    y_col: str = "매출수량",
    # open_rate smoothing
    alpha: float = 2.0,              # 라플라스 스무딩 강도(1~3 추천)
    # weather rolling windows
    w_short: int = 3,
    w_long: int = 7,
) -> pd.DataFrame:
    """
    1-step(=d 예측 시점에서 d-1까지 알고 있는 형태) 기준으로 계산 가능한 시계열 피처.
    제출 재귀 예측 로직(base에 pred를 채우며 하루씩 진행)과 학습 피처 구조를 동일하게 맞추기 위함.

    생성 피처:
    - lag_1, lag_7, lag_14, lag_28
    - rolling mean/std: shift(1) 기준 (d-1까지의 정보만)
    - menu_weekday_open_rate: 요일별 영업확률(>0 비율) [라플라스 스무딩]
    - days_since_last_sale: d-1 기준 마지막 판매일로부터 경과일
    - (추가) weather TS:
        temp_lag_1, temp_roll_mean_{3,7}
        rain_lag_1, rain_roll_mean_{3,7}
        is_rain_roll_mean_7
      * rain은 rain_log1p 있으면 그걸 쓰고, 없으면 log1p(rain_mm)
      * 모든 rolling은 shift(1) 기반 (누수 방지)
    """
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce").dt.normalize()
    out = out.sort_values([key_col, date_col]).reset_index(drop=True)

    # reference 준비(누수 방지: out 마지막 날짜까지만 참조)
    ref = train_reference[[key_col, date_col, y_col]].copy()
    ref[date_col] = pd.to_datetime(ref[date_col], errors="coerce").dt.normalize()
    ref[y_col] = pd.to_numeric(ref[y_col], errors="coerce").fillna(0).clip(lower=0).astype(np.float32)

    max_d = out[date_col].max()
    ref = ref[ref[date_col] <= max_d].copy()

    # weekday 보장 (open_rate 계산용)
    if "weekday" not in out.columns:
        out["weekday"] = out[date_col].dt.weekday.astype(np.int8)
    if "weekday" not in ref.columns:
        ref["weekday"] = ref[date_col].dt.weekday.astype(np.int8)

    grp = out[key_col].astype("string")
    y = pd.to_numeric(out[y_col], errors="coerce").fillna(0).clip(lower=0).astype(np.float32)

    # (1) sales lag
    for l in [1, 7, 14, 28]:
        out[f"lag_{l}"] = y.groupby(grp, sort=False).shift(l).fillna(0.0).astype(np.float32)

    # (2) sales rolling: shift(1) 기준
    y_shift = y.groupby(grp, sort=False).shift(1).fillna(0.0).astype(np.float32)

    out["roll_mean_7"] = (
        y_shift.groupby(grp, sort=False)
               .rolling(7, min_periods=1).mean()
               .reset_index(level=0, drop=True)
               .astype(np.float32)
    )
    out["roll_std_7"] = (
        y_shift.groupby(grp, sort=False)
               .rolling(7, min_periods=1).std()
               .reset_index(level=0, drop=True)
               .fillna(0.0)
               .astype(np.float32)
    )
    out["roll_mean_28"] = (
        y_shift.groupby(grp, sort=False)
               .rolling(28, min_periods=1).mean()
               .reset_index(level=0, drop=True)
               .astype(np.float32)
    )

    # ---------------------------------------------------------
    # (2-ADD) Weather TS (누수 방지: shift(1) 후 rolling)
    #   전제: out에 temp_avg, rain_mm, is_rain (그리고 가능하면 rain_log1p)
    #   없으면 0으로 안전 처리
    # ---------------------------------------------------------
    if "temp_avg" in out.columns:
        temp = pd.to_numeric(out["temp_avg"], errors="coerce").fillna(0.0).astype(np.float32)
    else:
        temp = pd.Series(np.zeros(len(out), dtype=np.float32), index=out.index)

    if "rain_log1p" in out.columns:
        rain = pd.to_numeric(out["rain_log1p"], errors="coerce").fillna(0.0).clip(lower=0.0).astype(np.float32)
    else:
        if "rain_mm" in out.columns:
            rain_mm = pd.to_numeric(out["rain_mm"], errors="coerce").fillna(0.0).clip(lower=0.0).astype(np.float32)
            rain = np.log1p(rain_mm).astype(np.float32)
        else:
            rain = pd.Series(np.zeros(len(out), dtype=np.float32), index=out.index)

    if "is_rain" in out.columns:
        is_r = pd.to_numeric(out["is_rain"], errors="coerce").fillna(0.0).clip(0.0, 1.0).astype(np.float32)
    else:
        is_r = pd.Series(np.zeros(len(out), dtype=np.float32), index=out.index)

    # shift(1)
    temp_shift = temp.groupby(grp, sort=False).shift(1).fillna(0.0).astype(np.float32)
    rain_shift = rain.groupby(grp, sort=False).shift(1).fillna(0.0).astype(np.float32)
    isr_shift  = is_r.groupby(grp, sort=False).shift(1).fillna(0.0).astype(np.float32)

    # lag 1
    out["temp_lag_1"] = temp_shift
    out["rain_lag_1"] = rain_shift

    # rolling means
    out[f"temp_roll_mean_{w_short}"] = (
        temp_shift.groupby(grp, sort=False)
                  .rolling(w_short, min_periods=1).mean()
                  .reset_index(level=0, drop=True)
                  .astype(np.float32)
    )
    out[f"temp_roll_mean_{w_long}"] = (
        temp_shift.groupby(grp, sort=False)
                  .rolling(w_long, min_periods=1).mean()
                  .reset_index(level=0, drop=True)
                  .astype(np.float32)
    )
    out[f"rain_roll_mean_{w_short}"] = (
        rain_shift.groupby(grp, sort=False)
                  .rolling(w_short, min_periods=1).mean()
                  .reset_index(level=0, drop=True)
                  .astype(np.float32)
    )
    out[f"rain_roll_mean_{w_long}"] = (
        rain_shift.groupby(grp, sort=False)
                  .rolling(w_long, min_periods=1).mean()
                  .reset_index(level=0, drop=True)
                  .astype(np.float32)
    )
    out["is_rain_roll_mean_7"] = (
        isr_shift.groupby(grp, sort=False)
                 .rolling(7, min_periods=1).mean()
                 .reset_index(level=0, drop=True)
                 .astype(np.float32)
    )

    # ---------------------------------------------------------
    # (3) 요일별 open_rate (ref 기반) - apply 제거 + 라플라스 스무딩
    # open_rate = (open_cnt + alpha) / (total_cnt + 2*alpha)
    # ---------------------------------------------------------
    ref2 = ref.copy()
    ref2["is_pos"] = (ref2[y_col] > 0).astype(np.int8)

    cnt = ref2.groupby([key_col, "weekday"], observed=False)["is_pos"].agg(["sum", "count"]).reset_index()
    cnt.rename(columns={"sum": "open_cnt", "count": "total_cnt"}, inplace=True)

    cnt["menu_weekday_open_rate"] = (
        (cnt["open_cnt"].astype(np.float32) + alpha) /
        (cnt["total_cnt"].astype(np.float32) + 2.0 * alpha)
    ).astype(np.float32)

    out = out.merge(cnt[[key_col, "weekday", "menu_weekday_open_rate"]], on=[key_col, "weekday"], how="left")
    out["menu_weekday_open_rate"] = out["menu_weekday_open_rate"].fillna(0.0).astype(np.float32)

    # (4) days_since_last_sale: shift(1) 기준
    last_sale_date = out[date_col].where(y > 0).groupby(grp, sort=False).ffill().shift(1)
    out["days_since_last_sale"] = (
        (out[date_col] - last_sale_date).dt.days.fillna(999).clip(0, 999).astype(np.int16)
    )

    # 수치 안정화
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

    return out


def qc_after_timeseries_features_step1(df, name="df", w_short=3, w_long=7):
    must = [
        "lag_1","lag_7","lag_14","lag_28",
        "roll_mean_7","roll_std_7","roll_mean_28",
        "menu_weekday_open_rate",
        "days_since_last_sale",
        # weather TS
        "temp_lag_1", f"temp_roll_mean_{w_short}", f"temp_roll_mean_{w_long}",
        "rain_lag_1", f"rain_roll_mean_{w_short}", f"rain_roll_mean_{w_long}",
        "is_rain_roll_mean_7",
    ]
    missing = [c for c in must if c not in df.columns]
    assert not missing, f"[{name}] missing step1 ts features: {missing}"
    assert df[must].isna().sum().sum() == 0, f"[{name}] NaNs exist"
    assert df["menu_weekday_open_rate"].between(0, 1).all(), f"[{name}] open_rate out of range"
    print(f"✅ timeseries-features STEP1 QC PASS: {name}")


# ✅ 적용 (train_f / tests_f 둘 다 step1로 통일)
train_f = add_timeseries_features_step1(train_f, train_reference=train)
tests_f = {k: add_timeseries_features_step1(v, train_reference=train) for k, v in tests_f.items()}

qc_after_timeseries_features_step1(train_f, "train")
for k, v in tests_f.items():
    qc_after_timeseries_features_step1(v, k)


✅ timeseries-features STEP1 QC PASS: train
✅ timeseries-features STEP1 QC PASS: TEST_00
✅ timeseries-features STEP1 QC PASS: TEST_01
✅ timeseries-features STEP1 QC PASS: TEST_02
✅ timeseries-features STEP1 QC PASS: TEST_03
✅ timeseries-features STEP1 QC PASS: TEST_04
✅ timeseries-features STEP1 QC PASS: TEST_05
✅ timeseries-features STEP1 QC PASS: TEST_06
✅ timeseries-features STEP1 QC PASS: TEST_07
✅ timeseries-features STEP1 QC PASS: TEST_08
✅ timeseries-features STEP1 QC PASS: TEST_09


# 5. 학습 데이터 구성
## 5.1 Time-based Split

In [13]:
DATE_COL = "영업일자"
KEY_COL  = "영업장명_메뉴명"
Y_COL    = "매출수량"

def time_split(df, val_days=28):
    df = df.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    max_date = df[DATE_COL].max()
    cut = max_date - pd.Timedelta(days=val_days)
    tr = df[df[DATE_COL] <= cut].copy()
    va = df[df[DATE_COL] >  cut].copy()
    return tr, va

train_tr, train_va = time_split(train_f, val_days=28)


## 5.2 Feature 컬럼 확정

In [14]:
DROP_COLS = [DATE_COL, Y_COL]
FEATURE_COLS = [c for c in train_f.columns if c not in DROP_COLS]

CAT_COLS = [c for c in ["영업장명", "메뉴명", KEY_COL] if c in FEATURE_COLS]

def cast_cat(df):
    df = df.copy()
    for c in CAT_COLS:
        df[c] = df[c].astype("category")
    return df

train_tr = cast_cat(train_tr)
train_va = cast_cat(train_va)

In [15]:
# =========================
# [CHECK-1] Weather is in model inputs?
# =========================
weather_cols = ["temp_avg","temp_min","temp_max","temp_range","rain_mm","rain_log1p","is_rain"]

print("Weather cols in train_f:", [c for c in weather_cols if c in train_f.columns])
print("Weather cols in FEATURE_COLS:", [c for c in weather_cols if c in FEATURE_COLS])

# 값 분산(상수면 사실상 무의미)
cols = [c for c in weather_cols if c in train_tr.columns]
if cols:
    print(train_tr[cols].describe().T[["mean","std","min","max"]])
else:
    print("❌ train_tr에 weather 컬럼이 없음 (학습에 안 들어감)")


Weather cols in train_f: ['temp_avg', 'temp_min', 'temp_max', 'temp_range', 'rain_mm', 'rain_log1p', 'is_rain']
Weather cols in FEATURE_COLS: ['temp_avg', 'temp_min', 'temp_max', 'temp_range', 'rain_mm', 'rain_log1p', 'is_rain']
              mean     std      min     max
temp_avg   11.7284  9.9466 -13.4000 30.9000
temp_min    7.0516 10.2755 -16.7000 26.5000
temp_max   17.0048 10.0589  -8.0000 34.9000
temp_range  9.9532  3.7415   1.2000 20.5000
rain_mm     3.3526 10.8228   0.0000 91.6000
rain_log1p  0.5018  1.0378   0.0000  4.5283
is_rain     0.2778  0.4479   0.0000  1.0000


# 6. 모델 학습 (Hurdle Model)
## 6.1 1단계 분류 (Sale / No Sale)
- 모델: LightGBM Classifier

In [16]:
import lightgbm as lgb
import numpy as np

train_tr["is_pos"] = (train_tr[Y_COL] > 0).astype(int)
train_va["is_pos"] = (train_va[Y_COL] > 0).astype(int)

clf = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

clf.fit(
    train_tr[FEATURE_COLS], train_tr["is_pos"],
    eval_set=[(train_va[FEATURE_COLS], train_va["is_pos"])],
    eval_metric="binary_logloss",
    categorical_feature=CAT_COLS,
    callbacks=[lgb.early_stopping(100, verbose=False)]
)


[LightGBM] [Info] Number of positive: 45930, number of negative: 51342
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004667 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5509
[LightGBM] [Info] Number of data points in the train set: 97272, number of used features: 44
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.472181 -> initscore=-0.111391
[LightGBM] [Info] Start training from score -0.111391


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.03
,n_estimators,2000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


## 6.2 2단계 회귀 (팔릴 때 수량)
- 모델: LightGBM Regressor

In [ ]:
# =========================================================
# [ON] Regressor (팔릴 때 수량) - 그대로 사용
# =========================================================
tr_pos = train_tr[train_tr[Y_COL] > 0].copy()
va_pos = train_va[train_va[Y_COL] > 0].copy()

tr_pos["y_log"] = np.log1p(tr_pos[Y_COL])
va_pos["y_log"] = np.log1p(va_pos[Y_COL])

reg = lgb.LGBMRegressor(
    n_estimators=4000,
    learning_rate=0.03,
    num_leaves=127,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
reg.fit(
    tr_pos[FEATURE_COLS], tr_pos["y_log"],
    eval_set=[(va_pos[FEATURE_COLS], va_pos["y_log"])],
    eval_metric="l2",
    categorical_feature=CAT_COLS,
    callbacks=[lgb.early_stopping(150, verbose=False)]
)

print("✅ trained ON models: clf / reg")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008309 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5090
[LightGBM] [Info] Number of data points in the train set: 45930, number of used features: 42
[LightGBM] [Info] Start training from score 2.093181
✅ trained ON models: clf / reg
[LightGBM] [Info] Number of positive: 45930, number of negative: 51342
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004806 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4395
[LightGBM] [Info] Number of data points in the train set: 97272, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.472181 -> initscore=-0.111391
[LightGBM] [Info] Start training from score -0.111391
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhea

# 7. Weather OFF 모델

In [ ]:
# =========================================================
# [OFF] Train a second model WITHOUT weather columns (OFF model)
# - MUST be trained on the same "final train_f" distribution
# - Used only for store-level ensemble
# =========================================================
weather_cols_all = ["temp_avg","temp_min","temp_max","temp_range","rain_mm","is_rain","rain_log1p"]

# 0) 최신 train_f에서 날씨만 제거한 버전 생성
train_f_off = train_f.drop(columns=[c for c in weather_cols_all if c in train_f.columns], errors="ignore")

# 1) OFF도 동일한 방식으로 time split
train_tr_off, train_va_off = time_split(train_f_off, val_days=28)

# 2) OFF feature/cat 정의 (OFF 기준으로 다시!)
DROP_COLS_OFF = [DATE_COL, Y_COL]
FEATURE_COLS_OFF = [c for c in train_f_off.columns if c not in DROP_COLS_OFF]
CAT_COLS_OFF = [c for c in ["영업장명", "메뉴명", KEY_COL] if c in FEATURE_COLS_OFF]

def cast_cat_off(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in CAT_COLS_OFF:
        out[c] = out[c].astype("category")
    return out

train_tr_off = cast_cat_off(train_tr_off)
train_va_off = cast_cat_off(train_va_off)

# 3) OFF classifier
train_tr_off["is_pos"] = (train_tr_off[Y_COL] > 0).astype(int)
train_va_off["is_pos"] = (train_va_off[Y_COL] > 0).astype(int)

clf_off = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
clf_off.fit(
    train_tr_off[FEATURE_COLS_OFF], train_tr_off["is_pos"],
    eval_set=[(train_va_off[FEATURE_COLS_OFF], train_va_off["is_pos"])],
    eval_metric="binary_logloss",
    categorical_feature=CAT_COLS_OFF,
    callbacks=[lgb.early_stopping(100, verbose=False)]
)

# 4) OFF regressor (양수만)
tr_pos_off = train_tr_off[train_tr_off[Y_COL] > 0].copy()
va_pos_off = train_va_off[train_va_off[Y_COL] > 0].copy()

tr_pos_off["y_log"] = np.log1p(tr_pos_off[Y_COL])
va_pos_off["y_log"] = np.log1p(va_pos_off[Y_COL])

reg_off = lgb.LGBMRegressor(
    n_estimators=4000,
    learning_rate=0.03,
    num_leaves=127,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
reg_off.fit(
    tr_pos_off[FEATURE_COLS_OFF], tr_pos_off["y_log"],
    eval_set=[(va_pos_off[FEATURE_COLS_OFF], va_pos_off["y_log"])],
    eval_metric="l2",
    categorical_feature=CAT_COLS_OFF,
    callbacks=[lgb.early_stopping(150, verbose=False)]
)

print("✅ trained OFF models: clf_off / reg_off")
print("OFF feature cols:", len(FEATURE_COLS_OFF), "| cat:", CAT_COLS_OFF)


# 8. 전역 규칙 / 통계 맵

In [18]:
# =========================================================
# [ADD] Global maps for inference tuning (run once)
# - 메뉴별 판매율(pos_rate) -> adaptive a, p_floor
# - 업장별 월평균 매출 -> store peak months
# =========================================================
import numpy as np
import pandas as pd

DATE_COL = "영업일자"
KEY_COL  = "영업장명_메뉴명"
Y_COL    = "매출수량"

def build_menu_gate_maps(train_f: pd.DataFrame, key_col=KEY_COL, y_col=Y_COL):
    # 메뉴별 판매율(>0 비율)
    pos_rate = (
        train_f.assign(is_pos=(train_f[y_col] > 0).astype(int))
              .groupby(key_col)["is_pos"].mean()
    )

    # 메뉴별 adaptive a: 희소 메뉴일수록 a 높게(under-predict 완화)
    # (범위는 안전하게)
    def adaptive_a(p):
        return float(np.clip(0.35 + 0.55 * (1.0 - p), 0.35, 0.85))

    # 메뉴별 p_floor: 희소 메뉴일수록 p를 더 잘라서 "미세 양수" 억제
    def adaptive_p_floor(p):
        return float(np.clip(0.12 + 0.55 * (1.0 - p), 0.12, 0.80))

    menu_a_map = {k: adaptive_a(v) for k, v in pos_rate.items()}
    menu_pfloor_map = {k: adaptive_p_floor(v) for k, v in pos_rate.items()}
    return pos_rate, menu_a_map, menu_pfloor_map

def build_store_peak_month_map(train_f: pd.DataFrame, store_col="영업장명", date_col=DATE_COL, y_col=Y_COL, top_k=3):
    df = train_f.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce").dt.normalize()
    df["month"] = df[date_col].dt.month.astype(np.int8)

    # 업장-월 평균 매출(일 단위 평균이 아니라, 해당 업장-월의 전체 행 평균; 패딩된 데이터라면 일/메뉴 포함)
    # 실무적으로는 "업장 일매출"로 만드는 게 더 안정적이라 아래처럼 집계해서 사용
    daily = (
        df.groupby([store_col, date_col], as_index=False)[y_col].sum()
          .assign(month=lambda x: x[date_col].dt.month.astype(np.int8))
    )
    store_month_mean = (
        daily.groupby([store_col, "month"])[y_col].mean().reset_index()
    )

    peak_map = {}
    for store, g in store_month_mean.groupby(store_col):
        top_months = g.sort_values(y_col, ascending=False)["month"].head(top_k).tolist()
        peak_map[store] = set(int(m) for m in top_months)

    return peak_map

# --- build maps (FIXED: use train_f limited to train range, not train_tr itself) ---

# 1) split 기준: train_tr의 마지막 날짜(= train 구간 끝)로 컷
cut_date = pd.to_datetime(train_tr[DATE_COL].max()).normalize()

# 2) train_f에서 train 구간까지만 사용 (val 누수 방지 + 안정)
train_for_maps = train_f[pd.to_datetime(train_f[DATE_COL]).dt.normalize() <= cut_date].copy()

POS_RATE, MENU_A_MAP, MENU_PFLOOR_MAP = build_menu_gate_maps(train_for_maps)
STORE_PEAK_MONTHS = build_store_peak_month_map(train_for_maps, top_k=3)

print("✅ built maps (FIXED):",
      "cut_date:", cut_date,
      "rows_used:", len(train_for_maps),
      "menu_a:", len(MENU_A_MAP),
      "menu_p_floor:", len(MENU_PFLOOR_MAP),
      "store_peaks:", len(STORE_PEAK_MONTHS))


✅ built maps (FIXED): cut_date: 2024-05-18 00:00:00 rows_used: 97272 menu_a: 193 menu_p_floor: 193 store_peaks: 9


# 9. 성수기 반영 및 재학습

In [19]:
# =========================================================
# [FIX] STORE_PEAK_MONTHS 반영해서 "date features만" 재생성
# - 이미 menu/store/ts 피처가 만들어져 있으므로, date 관련 컬럼만 덮어쓰기
# - 학습/검증/추론 정합을 위해 반드시 수행
# =========================================================

def refresh_date_features_only(df, store_peak_months):
    keep = [c for c in df.columns if c not in [
        "year","month","day","weekday","is_weekend",
        "day_of_year","sin_dow","cos_dow",
        "is_holiday","days_to_next_holiday","days_from_prev_holiday",
        "is_holiday_near_1","is_holiday_near_2",
        "store_is_peak_month"
    ]]
    base = df[keep].copy()
    # date feature 재생성 (필요 컬럼은 basic_preprocess로 이미 있음)
    dated = add_date_features(base, store_peak_months=store_peak_months)
    return dated

train_f = refresh_date_features_only(train_f, STORE_PEAK_MONTHS)
tests_f = {k: refresh_date_features_only(v, STORE_PEAK_MONTHS) for k, v in tests_f.items()}

print("✅ refreshed date features with STORE_PEAK_MONTHS")


✅ refreshed date features with STORE_PEAK_MONTHS


In [20]:
# =========================================================
# [RE-TRAIN] peak month 반영된 train_f 기준으로 재분할/재학습
# =========================================================
# 1) split 재생성
train_tr, train_va = time_split(train_f, val_days=28)

# 2) feature columns 재확정 (store_is_peak_month 포함되도록)
DROP_COLS = [DATE_COL, Y_COL]
FEATURE_COLS = [c for c in train_f.columns if c not in DROP_COLS]
CAT_COLS = [c for c in ["영업장명", "메뉴명", KEY_COL] if c in FEATURE_COLS]

def cast_cat(df):
    df = df.copy()
    for c in CAT_COLS:
        df[c] = df[c].astype("category")
    return df

train_tr = cast_cat(train_tr)
train_va = cast_cat(train_va)

# 3) 분류기 재학습
train_tr["is_pos"] = (train_tr[Y_COL] > 0).astype(int)
train_va["is_pos"] = (train_va[Y_COL] > 0).astype(int)

clf = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
clf.fit(
    train_tr[FEATURE_COLS], train_tr["is_pos"],
    eval_set=[(train_va[FEATURE_COLS], train_va["is_pos"])],
    eval_metric="binary_logloss",
    categorical_feature=CAT_COLS,
    callbacks=[lgb.early_stopping(100, verbose=False)]
)

# 4) 회귀기 재학습 (양수만)
tr_pos = train_tr[train_tr[Y_COL] > 0].copy()
va_pos = train_va[train_va[Y_COL] > 0].copy()
tr_pos["y_log"] = np.log1p(tr_pos[Y_COL])
va_pos["y_log"] = np.log1p(va_pos[Y_COL])

reg = lgb.LGBMRegressor(
    n_estimators=4000,
    learning_rate=0.03,
    num_leaves=127,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
reg.fit(
    tr_pos[FEATURE_COLS], tr_pos["y_log"],
    eval_set=[(va_pos[FEATURE_COLS], va_pos["y_log"])],
    eval_metric="l2",
    categorical_feature=CAT_COLS,
    callbacks=[lgb.early_stopping(150, verbose=False)]
)

# 5) 전역 맵은 train_tr 기준으로 재생성 (val 누수 방지)
# POS_RATE, MENU_A_MAP, MENU_PFLOOR_MAP = build_menu_gate_maps(train_tr)
# STORE_PEAK_MONTHS = build_store_peak_month_map(train_tr, top_k=3)

print("✅ retrained models and rebuilt maps with peak-month feature included")

# =========================================================
# [ADD] Store-level weather ensemble weights (from your DELTA table)
# =========================================================
STORE_WEATHER_WEIGHT = {
    "포레스트릿": 0.85,
    "화담숲카페": 0.85,
    "화담숲주막": 0.65,
    "카페테리아": 0.50,
    "연회장": 0.50,
    "담하": 0.50,
    "미라시아": 0.30,
    "라그로타": 0.30,
    "느티나무 셀프BBQ": 0.25,
}



[LightGBM] [Info] Number of positive: 45930, number of negative: 51342
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007522 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5511
[LightGBM] [Info] Number of data points in the train set: 97272, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.472181 -> initscore=-0.111391
[LightGBM] [Info] Start training from score -0.111391
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012153 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5092
[LightGBM] [Info] Number of data points in the train set: 45930, number of used features: 43
[LightGBM] [Info] Start training from score 2.093181
✅ retrained models and rebuilt maps with peak-month feature included


# 10. 검증 & 파라미터 튜닝

In [21]:
# =========================
# [OVERRIDE] SMAPE helpers + predict_hurdle (B안: 선형 게이트) + a 스윕
# - 이 셀(블록)만 기존 8번 블록과 "통째로" 교체해서 실행
# - 전제: cast_cat, FEATURE_COLS, KEY_COL, Y_COL, train_va, clf, reg 가 이미 위에서 정의/학습되어 있어야 함
# =========================

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1) SMAPE: 전체 + 업장별
# ---------------------------------------------------------
def smape_overall_and_by_store(
    df: pd.DataFrame,
    store_col: str = "영업장명",
    item_col: str = "영업장명_메뉴명",
    actual_col: str = "매출수량",
    pred_col: str = "pred",
    eps: float = 1e-6,
):
    store_rows = []
    store_scores = []

    for store, gs in df.groupby(store_col):
        item_scores = []

        for _, gi in gs.groupby(item_col):
            g = gi[gi[actual_col] != 0]  # ✅ Ti: 실제 매출 있는 날만 평가
            if len(g) == 0:
                continue

            A = g[actual_col].values
            P = g[pred_col].values
            sm = np.mean(2 * np.abs(A - P) / (np.abs(A) + np.abs(P) + eps))
            item_scores.append(sm)

        if item_scores:
            store_sm = float(np.mean(item_scores))
            store_rows.append({"영업장명": store, "SMAPE": store_sm, "n_items": len(item_scores)})
            store_scores.append(store_sm)

    overall = float(np.mean(store_scores)) if store_scores else 0.0
    store_df = (
        pd.DataFrame(store_rows).sort_values("SMAPE")
        if store_rows else pd.DataFrame(columns=["영업장명", "SMAPE", "n_items"])
    )
    return overall, store_df
# =========================
# [EXPERIMENT] Weather ON vs OFF (retrain + same metric)
# =========================
import lightgbm as lgb
import numpy as np
import pandas as pd

DATE_COL = "영업일자"
KEY_COL  = "영업장명_메뉴명"
Y_COL    = "매출수량"

weather_cols_all = ["temp_avg","temp_min","temp_max","temp_range","rain_mm","is_rain","rain_log1p"]

def _train_eval_once(df_full, seed=42):
    # 1) split
    tr, va = time_split(df_full, val_days=28)

    # 2) features
    drop_cols = [DATE_COL, Y_COL]
    feature_cols = [c for c in df_full.columns if c not in drop_cols]
    cat_cols = [c for c in ["영업장명", "메뉴명", KEY_COL] if c in feature_cols]

    def cast_cat_local(df):
        df = df.copy()
        for c in cat_cols:
            df[c] = df[c].astype("category")
        return df

    tr = cast_cat_local(tr); va = cast_cat_local(va)

    # 3) hurdle models
    tr["is_pos"] = (tr[Y_COL] > 0).astype(int)
    va["is_pos"] = (va[Y_COL] > 0).astype(int)

    clf = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, random_state=seed
    )
    clf.fit(
        tr[feature_cols], tr["is_pos"],
        eval_set=[(va[feature_cols], va["is_pos"])],
        eval_metric="binary_logloss",
        categorical_feature=cat_cols,
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )

    tr_pos = tr[tr[Y_COL] > 0].copy()
    va_pos = va[va[Y_COL] > 0].copy()
    tr_pos["y_log"] = np.log1p(tr_pos[Y_COL])
    va_pos["y_log"] = np.log1p(va_pos[Y_COL])

    reg = lgb.LGBMRegressor(
        n_estimators=4000, learning_rate=0.03, num_leaves=127,
        subsample=0.8, colsample_bytree=0.8, random_state=seed
    )
    reg.fit(
        tr_pos[feature_cols], tr_pos["y_log"],
        eval_set=[(va_pos[feature_cols], va_pos["y_log"])],
        eval_metric="l2",
        categorical_feature=cat_cols,
        callbacks=[lgb.early_stopping(150, verbose=False)]
    )

    # 4) predict (너의 기본 선형게이트, adaptive map 없이 고정 a=0.55로 비교)
    Xva = cast_cat_local(va)[feature_cols]
    p = np.clip(clf.predict_proba(Xva)[:, 1].astype(np.float32), 0.02, 0.98)
    y_pos = np.clip(np.expm1(reg.predict(Xva).astype(np.float32)), 0.0, None)

    va2 = va.copy()
    va2["pred"] = y_pos * (0.55 + 0.45 * p)

    overall, _ = smape_overall_and_by_store(va2)
    return overall

def compare_weather_on_off(train_f):
    # OFF: 날씨 컬럼 제거
    no_w = train_f.drop(columns=[c for c in weather_cols_all if c in train_f.columns], errors="ignore")
    s_off = _train_eval_once(no_w, seed=42)

    # ON: 그대로
    s_on = _train_eval_once(train_f, seed=42)

    print("SMAPE (weather OFF):", round(s_off, 6))
    print("SMAPE (weather ON ):", round(s_on, 6))
    print("Delta (ON - OFF)   :", round(s_on - s_off, 6))

compare_weather_on_off(train_f)


# ---------------------------------------------------------
# 2) predict_hurdle (B안): Hard Cutoff/PowerGate 대신 "선형 게이트"
#    gate = a + (1-a)*p  (p=0이어도 a만큼 남겨서 under-predict 완화)
# ---------------------------------------------------------
import numpy as np
import pandas as pd

def predict_hurdle(
    df: pd.DataFrame,
    a: float = 0.55,
    power: float = 1.0,
    p_clip=(0.02, 0.98),
    key_col: str = "영업장명_메뉴명",
    use_adaptive: bool = True,
    hard_cut: bool = True,
):
    X = cast_cat(df)[FEATURE_COLS]

    # 1) 팔릴 확률
    p = clf.predict_proba(X)[:, 1].astype(np.float32)
    p = np.clip(p, p_clip[0], p_clip[1])

    # ✅ 메뉴별 p_floor hard cut (미세양수 억제)
    if hard_cut:
        if use_adaptive and ("MENU_PFLOOR_MAP" in globals()):
            pfloor = df[key_col].map(MENU_PFLOOR_MAP).fillna(0.30).astype(np.float32).to_numpy()
        else:
            pfloor = np.full(len(df), 0.30, dtype=np.float32)
        p = np.where(p < pfloor, 0.0, p).astype(np.float32)

    # 2) 팔릴 때 수량
    y_log = reg.predict(X).astype(np.float32)
    y_pos = np.expm1(y_log).astype(np.float32)
    y_pos = np.clip(y_pos, 0.0, None)

    # (선택) 수량 스케일 보정
    if power != 1.0:
        y_pos = np.power(y_pos, power)

    # ✅ 3) adaptive a + 선형 게이트
    if use_adaptive and ("MENU_A_MAP" in globals()):
        a_vec = df[key_col].map(MENU_A_MAP).fillna(a).astype(np.float32).to_numpy()
    else:
        a_vec = np.full(len(df), a, dtype=np.float32)

    gate = a_vec + (1.0 - a_vec) * p
    y_hat = y_pos * gate

    return np.clip(y_hat, 0.0, None)



# ---------------------------------------------------------
# 3) [RUN] 단일 설정 확인
# ---------------------------------------------------------
va = train_va.copy()
va["pred"] = predict_hurdle(va, a=0.55, power=1.0)

overall_smape, store_smape_df = smape_overall_and_by_store(va)
print("Overall SMAPE:", overall_smape)
print(store_smape_df)
print("pred zeros ratio:", float((va["pred"] == 0).mean()))
print("pred min/max/mean:", float(va["pred"].min()), float(va["pred"].max()), float(va["pred"].mean()))


# ---------------------------------------------------------
# 4) [TUNE] a sweep + best store report
# ---------------------------------------------------------
results = []

for a in [0.35, 0.45, 0.55, 0.65, 0.75]:
    va = train_va.copy()
    va["pred"] = predict_hurdle(va, a=a, power=1.0)
    overall, store_df = smape_overall_and_by_store(va)

    zeros = float((va["pred"] == 0).mean())
    results.append((a, overall, zeros, store_df))

    print(f"a={a:.2f}  overall={overall:.4f}  zeros={zeros:.4f}")

best = min(results, key=lambda x: x[1])
best_a, best_overall, best_zeros, best_store_df = best

print("\n====================")
print(f"BEST a = {best_a:.2f}")
print(f"BEST overall = {best_overall:.4f}")
print(f"BEST zeros = {best_zeros:.4f}")
print("====================\n")

print(best_store_df)


[LightGBM] [Info] Number of positive: 45930, number of negative: 51342
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008330 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4397
[LightGBM] [Info] Number of data points in the train set: 97272, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.472181 -> initscore=-0.111391
[LightGBM] [Info] Start training from score -0.111391
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004186 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4017
[LightGBM] [Info] Number of data points in the train set: 45930, number of used features: 36
[LightGBM] [Info] Start training from score 2.093181


C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\510474532.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for store, gs in df.groupby(store_col):
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\510474532.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for _, gi in gs.groupby(item_col):


[LightGBM] [Info] Number of positive: 45930, number of negative: 51342
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007558 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5511
[LightGBM] [Info] Number of data points in the train set: 97272, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.472181 -> initscore=-0.111391
[LightGBM] [Info] Start training from score -0.111391
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016833 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5092
[LightGBM] [Info] Number of data points in the train set: 45930, number of used features: 43
[LightGBM] [Info] Start training from score 2.093181


C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\510474532.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for store, gs in df.groupby(store_col):
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\510474532.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for _, gi in gs.groupby(item_col):


SMAPE (weather OFF): 0.483364
SMAPE (weather ON ): 0.467079
Delta (ON - OFF)   : -0.016285


...

a=0.55  overall=0.4598  zeros=0.0000


C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\510474532.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for store, gs in df.groupby(store_col):
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\510474532.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for _, gi in gs.groupby(item_col):


a=0.65  overall=0.4598  zeros=0.0000


C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\510474532.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for store, gs in df.groupby(store_col):
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\510474532.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for _, gi in gs.groupby(item_col):


a=0.75  overall=0.4598  zeros=0.0000

BEST a = 0.35
BEST overall = 0.4598
BEST zeros = 0.0000

         영업장명  SMAPE  n_items
7       화담숲주막 0.3270        8
8       화담숲카페 0.3485        5
4         연회장 0.3916       23
3        미라시아 0.4500       31
6       포레스트릿 0.4818       11
2        라그로타 0.4910       25
0  느티나무 셀프BBQ 0.5352       23
1          담하 0.5433       40
5       카페테리아 0.5698       14


# 11. 추론 파이프라인 (Inference)
## 11.1 미래 예측 구조 정의

In [ ]:
import numpy as np
import pandas as pd

# 미래 7일 프레임 확장
def expand_future_7days(df_28, key_col="영업장명_메뉴명", date_col="영업일자", y_col="매출수량", horizon=7):
    hist = df_28.copy()
    hist[date_col] = pd.to_datetime(hist[date_col], errors="coerce").dt.normalize()
    hist = hist[hist[date_col].notna()].copy()

    hist[y_col] = pd.to_numeric(hist[y_col], errors="coerce").fillna(0.0).clip(lower=0.0)
    hist = hist.groupby([key_col, date_col], as_index=False)[y_col].sum()

    last_date = hist[date_col].max()
    future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=horizon, freq="D")

    keys = np.sort(hist[key_col].unique())
    future = pd.MultiIndex.from_product([keys, future_dates], names=[key_col, date_col]).to_frame(index=False)
    future[y_col] = np.nan

    out = pd.concat([hist, future], ignore_index=True).sort_values([key_col, date_col]).reset_index(drop=True)
    return out, list(future_dates)


C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return name.str.contains(pat, regex=True, na=False)
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:56: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  set_hint = m_norm.str.contains(r"(?:^| )(세트|코스|정식|패키지|모둠|플래터|콤보|단체|set|course|combo|platter)(?: |$)",
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return name.str.contains(pat, regex=True, na=False)
C:\Users\DS3\AppData\Local\Temp\ipykernel_15676\3696639132.py:56: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.e

✅ 제출 파일 생성 완료: hurdle_linear_gate_submission_store_ensemble.csv


## 11.2  재귀 예측 로직

In [ ]:
# ---------------------------------------------------------
# [B] 피처 생성: 학습/추론 구조 통일 (STEP1 사용)
#   - 여기서 weather는 항상 붙여둔다 (ON 모델에 필요)
#   - OFF 모델은 예측 시 feature subset(FEATURE_COLS_OFF)만 사용하므로 OK
# ---------------------------------------------------------
def make_features(base_df, train_reference, store_history_df):
    x = base_df.copy()

    if "영업장명" not in x.columns or "메뉴명" not in x.columns:
        split = x["영업장명_메뉴명"].astype(str).str.split("_", n=1, expand=True)
        x["영업장명"] = split[0]
        x["메뉴명"] = split[1]

    # 날짜
    x = add_date_features(x, store_peak_months=STORE_PEAK_MONTHS)

    # 날씨 (관측 merge + 미래는 climatology로 자동 채움)
    x = add_weather_features(x, weather=WEATHER_DAILY, climatology=WEATHER_CLIM)

    # 메뉴/업장/시계열
    x = add_menu_features(x)
    x = add_store_features(x, history_df=store_history_df)
    x = add_timeseries_features_step1(x, train_reference=train_reference)

    return x


# 포레스트릿 룰

def apply_forestreet_rules(day_rows: pd.DataFrame, pred: np.ndarray) -> np.ndarray:
    need_cols = ["영업장명", "store_is_closed", "menu_weekday_open_rate", "days_since_last_sale"]
    for c in need_cols:
        if c not in day_rows.columns:
            return pred

    is_forest = (day_rows["영업장명"] == "포레스트릿")
    cut = (
        (day_rows["store_is_closed"] == 1) |
        (day_rows["menu_weekday_open_rate"] < 0.30) |
        (day_rows["days_since_last_sale"] > 14)
    )

    pred = pred.copy()
    pred[is_forest & cut] = 0.0
    pred[is_forest & (pred < 0.30)] = 0.0
    return pred


# ---------------------------------------------------------
# [C-ADD] Hurdle predictors (ON / OFF) + store ensemble
#   - ON : clf/reg + FEATURE_COLS
#   - OFF: clf_off/reg_off + FEATURE_COLS_OFF
# ---------------------------------------------------------
def predict_hurdle_on(
    df: pd.DataFrame,
    a: float = 0.55,
    power: float = 1.0,
    p_clip=(0.02, 0.98),
    key_col: str = "영업장명_메뉴명",
    use_adaptive: bool = True,
    hard_cut: bool = True,
):
    X = cast_cat(df)[FEATURE_COLS]

    p = clf.predict_proba(X)[:, 1].astype(np.float32)
    p = np.clip(p, p_clip[0], p_clip[1])

    if hard_cut:
        if use_adaptive and ("MENU_PFLOOR_MAP" in globals()):
            pfloor = df[key_col].map(MENU_PFLOOR_MAP).fillna(0.30).astype(np.float32).to_numpy()
        else:
            pfloor = np.full(len(df), 0.30, dtype=np.float32)
        p = np.where(p < pfloor, 0.0, p).astype(np.float32)

    y_log = reg.predict(X).astype(np.float32)
    y_pos = np.expm1(y_log).astype(np.float32)
    y_pos = np.clip(y_pos, 0.0, None)

    if power != 1.0:
        y_pos = np.power(y_pos, power)

    if use_adaptive and ("MENU_A_MAP" in globals()):
        a_vec = df[key_col].map(MENU_A_MAP).fillna(a).astype(np.float32).to_numpy()
    else:
        a_vec = np.full(len(df), a, dtype=np.float32)

    gate = a_vec + (1.0 - a_vec) * p
    y_hat = y_pos * gate
    return np.clip(y_hat, 0.0, None)


def predict_hurdle_off(
    df: pd.DataFrame,
    a: float = 0.55,
    power: float = 1.0,
    p_clip=(0.02, 0.98),
    key_col: str = "영업장명_메뉴명",
    use_adaptive: bool = True,
    hard_cut: bool = True,
):
    # OFF용 X 구성
    X = df.copy()
    for c in CAT_COLS_OFF:
        if c in X.columns:
            X[c] = X[c].astype("category")
    X = X[FEATURE_COLS_OFF]

    p = clf_off.predict_proba(X)[:, 1].astype(np.float32)
    p = np.clip(p, p_clip[0], p_clip[1])

    if hard_cut:
        if use_adaptive and ("MENU_PFLOOR_MAP" in globals()):
            pfloor = df[key_col].map(MENU_PFLOOR_MAP).fillna(0.30).astype(np.float32).to_numpy()
        else:
            pfloor = np.full(len(df), 0.30, dtype=np.float32)
        p = np.where(p < pfloor, 0.0, p).astype(np.float32)

    y_log = reg_off.predict(X).astype(np.float32)
    y_pos = np.expm1(y_log).astype(np.float32)
    y_pos = np.clip(y_pos, 0.0, None)

    if power != 1.0:
        y_pos = np.power(y_pos, power)

    if use_adaptive and ("MENU_A_MAP" in globals()):
        a_vec = df[key_col].map(MENU_A_MAP).fillna(a).astype(np.float32).to_numpy()
    else:
        a_vec = np.full(len(df), a, dtype=np.float32)

    gate = a_vec + (1.0 - a_vec) * p
    y_hat = y_pos * gate
    return np.clip(y_hat, 0.0, None)


def ensemble_pred_by_store(day_rows: pd.DataFrame, pred_on: np.ndarray, pred_off: np.ndarray) -> np.ndarray:
    w = (
        day_rows["영업장명"]
        .map(STORE_WEATHER_WEIGHT)
        .fillna(0.5)
        .astype(np.float32)
        .to_numpy()
    )
    return w * pred_on + (1.0 - w) * pred_off

# [D] 재귀 예측(7일): 하루씩 채움 + 업장 단위 앙상블
def recursive_predict_7days_hurdle(
    df_test_28: pd.DataFrame,
    train_reference: pd.DataFrame,
    a_best: float,
    key_col="영업장명_메뉴명",
    date_col="영업일자",
    y_col="매출수량",
):
    base, future_dates = expand_future_7days(df_test_28, key_col, date_col, y_col, horizon=7)

    # split 보장
    if "영업장명" not in base.columns or "메뉴명" not in base.columns:
        split = base[key_col].astype(str).str.split("_", n=1, expand=True)
        base["영업장명"] = split[0]
        base["메뉴명"] = split[1]

    base[date_col] = pd.to_datetime(base[date_col], errors="coerce").dt.normalize()
    base[y_col] = pd.to_numeric(base[y_col], errors="coerce").astype(float)

    for d in future_dates:
        d = pd.to_datetime(d).normalize()

        # 업장 운영 피처용 최근 히스토리: d-1까지의 base(실제+예측 채워진 값)
        hist_upto = base[base[date_col] < d].copy()
        hist_upto[y_col] = pd.to_numeric(hist_upto[y_col], errors="coerce").fillna(0.0).clip(lower=0.0)

        feat = make_features(
            base_df=base,
            train_reference=train_reference,   # 장기 통계(open_rate 등)
            store_history_df=hist_upto         # 최근 운영상태 반영
        )

        mask = (pd.to_datetime(feat[date_col]).dt.normalize() == d)
        day_rows = feat.loc[mask].copy()

        # ✅ 업장 단위 weather ON/OFF 앙상블
        pred_on  = predict_hurdle_on(day_rows, a=a_best, power=1.0, use_adaptive=True, hard_cut=True)
        pred_off = predict_hurdle_off(day_rows, a=a_best, power=1.0, use_adaptive=True, hard_cut=True)

        pred = ensemble_pred_by_store(day_rows, pred_on=pred_on, pred_off=pred_off)
        pred = np.clip(pred, 0.0, None)

        # (선택) 포레스트릿 룰은 앙상블 결과에 적용
        pred = apply_forestreet_rules(day_rows, pred)

        idx = base.index[(base[date_col] == d)]
        base.loc[idx, y_col] = pred  # 소수 유지

    future_only = base[base[date_col].isin([pd.to_datetime(x).normalize() for x in future_dates])].copy()

    # 최종 제출 정수화
    future_only[y_col] = np.where(future_only[y_col] < 0.5, 0, np.rint(future_only[y_col])).astype(np.int32)
    return future_only

## 11.3 제출 파일 생성 및 최종 실행

In [ ]:
# [E] sample_submission(wide) 포맷으로 채우기

def build_submission_hurdle(
    sample_path: str,
    tests_raw_dict: dict,
    train_reference: pd.DataFrame,
    a_best: float,
):
    sub = pd.read_csv(sample_path)
    key_cols = [c for c in sub.columns if c != "영업일자"]

    for test_name, df_test in tests_raw_dict.items():
        future_pred = recursive_predict_7days_hurdle(
            df_test_28=df_test,
            train_reference=train_reference,
            a_best=a_best,
            key_col="영업장명_메뉴명",
            date_col="영업일자",
            y_col="매출수량",
        )

        piv = future_pred.pivot(index="영업일자", columns="영업장명_메뉴명", values="매출수량")
        piv = piv.reindex(columns=key_cols).fillna(0.0)

        dates_sorted = sorted(piv.index)
        for i, dt in enumerate(dates_sorted, start=1):
            row_key = f"{test_name}+{i}일"
            sub.loc[sub["영업일자"] == row_key, key_cols] = piv.loc[dt, key_cols].to_numpy()

    return sub


# [RUN] 실행

best_a = 0.55  # 네 스윕에서 뽑은 값으로 고정

final_sub = build_submission_hurdle(
    sample_path="./data/sample_submission.csv",
    tests_raw_dict=tests,
    train_reference=train,  # pad_calendar된 train 권장
    a_best=best_a,
)

final_sub.to_csv("hurdle_linear_gate_submission_store_ensemble.csv", index=False, encoding="utf-8-sig")
print("✅ 제출 파일 생성 완료: hurdle_linear_gate_submission_store_ensemble.csv")